In [2]:
# Loads up the imports

from pathlib import Path
import random

import pandas as pd

In [3]:
# Sets up the constants

DATA_DIR = Path("data")
TRAIN_DIR = DATA_DIR / "training"
CLASSES = ["0", "1", "2", "3"]
VALIDATION_FRACTION = 0.20
SEED = 2100

In [4]:
# Load images and create a DataFrame with their paths, classes, and patient IDs.

image_records = []

for class_label in CLASSES:
    class_folder = TRAIN_DIR / class_label
    image_paths = sorted(class_folder.glob("*.png"))

    for image_path in image_paths:
        image_records.append(
            {
                "path": image_path,
                "label": int(class_label),
                "patient": image_path.name.split("_", maxsplit=1)[0],
            }
        )

training_data = pd.DataFrame(image_records)

print(f"Imagens carregadas: {len(training_data)}")
print(f"Pacientes na base: {training_data['patient'].nunique()}")

Imagens carregadas: 593
Pacientes na base: 20


In [5]:
# Split each class by patient so images from one patient cannot enter both sets.
# Select 20% of that class's patients for validation; with five patients, this selects one.
# A fixed seed makes the random patient selection reproducible.

random_generator = random.Random(SEED)
validation_patients = set()

for class_label, class_data in training_data.groupby("label", sort=True):
    class_patients = sorted(class_data["patient"].unique())
    validation_count = round(len(class_patients) * VALIDATION_FRACTION)
    selected_validation_patients = random_generator.sample(class_patients, validation_count)
    validation_patients.update(selected_validation_patients)
    print(
        f"Classe {class_label}: pacientes selecionados para validação: "
        f"{selected_validation_patients}"
    )

# Put all images from selected patients in validation; keep all remaining patients in training.

validation_mask = training_data["patient"].isin(validation_patients)
validation_data = training_data.loc[validation_mask].copy().reset_index(drop=True)
train_data = training_data.loc[~validation_mask].copy().reset_index(drop=True)

# Show how many images were assigned to each partition.

print(f"Pacientes de validação: {sorted(validation_patients)}")
print(f"Imagens para treino: {len(train_data)}")
print(f"Imagens para validação: {len(validation_data)}")

Classe 0: pacientes selecionados para validação: ['18']
Classe 1: pacientes selecionados para validação: ['24']
Classe 2: pacientes selecionados para validação: ['04']
Classe 3: pacientes selecionados para validação: ['19']
Pacientes de validação: ['04', '18', '19', '24']
Imagens para treino: 475
Imagens para validação: 118


In [6]:
# Combine both partitions so they can be summarized in one table.

all_data = pd.concat(
    [
        train_data.assign(split="Treino"),
        validation_data.assign(split="Validação"),
    ],
    ignore_index=True,
)

# Count images and distinct patients for each partition and class.

split_summary = (
    all_data.groupby(["split", "label"])
    .agg(images=("path", "size"), patients=("patient", "nunique"))
)

# Display the counts to check the resulting split.

split_summary

images  patients
split     label                  
Treino    0         118         4
          1         117         4
          2         120         4
          3         120         4
Validação 0          28         1
          1          30         1
          2          30         1
          3          30         1